In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [3]:
# === 0. Setup ===
import os
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict
from datetime import timedelta

# Optional (for torch Dataset skeleton – 학습 단계에서 유용)
try:
    import torch
    from torch.utils.data import Dataset
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False
    print("[Info] PyTorch not installed. You can still run up to fold-splitting.")

@dataclass
class Config:
    lookback: int = 28   # L
    horizon: int  = 7    # H
    patch_len: int = 7   # for PatchTST later
    stride: int    = 1   # for PatchTST later
    n_splits: int  = 5   # K-fold
    embargo_days: int = 35  # purge gap around validation (≈ lookback + horizon)
    seed: int = 42

CFG = Config()
np.random.seed(CFG.seed)

print(CFG)

Config(lookback=28, horizon=7, patch_len=7, stride=1, n_splits=5, embargo_days=35, seed=42)


In [4]:
# === 1. Load & sort ===
# 경로는 ipynb 기준으로 맞춰주세요.
TRAIN_PATH = "./data_filtering/filtered/train.csv"  # ex) "./dataset/train/train.csv"

df = pd.read_csv(TRAIN_PATH)

# Basic parsing
df['date'] = pd.to_datetime(df['date'])
# 안전장치: store_menu 없으면 생성
if 'store_menu' not in df.columns:
    df['store_menu'] = df['store'].astype(str) + "_" + df['menu'].astype(str)

# 정렬 & 타입 정리
df = df.sort_values(['store_menu', 'date']).reset_index(drop=True)

print("Rows:", len(df))
print("Unique store_menu:", df['store_menu'].nunique())
print(df.head(3))


Rows: 102676
Unique store_menu: 193
   date_ordinal       date          store_menu       store     menu  sales
0        738521 2023-01-01  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트      0
1        738522 2023-01-02  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트      0
2        738523 2023-01-03  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트      0


In [5]:
# === 2. Feature engineering (calendar only, rule-safe) ===
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame.copy()
    f['dow'] = f['date'].dt.weekday           # 0..6
    f['dom'] = f['date'].dt.day               # 1..31
    f['month'] = f['date'].dt.month           # 1..12
    f['is_weekend'] = (f['dow'] >= 5).astype(int)

    # Sine/Cos encoding for weekly seasonality
    f['dow_sin'] = np.sin(2 * np.pi * f['dow'] / 7)
    f['dow_cos'] = np.cos(2 * np.pi * f['dow'] / 7)
    return f

df_feat = add_calendar_features(df)

# 사용할 입력 피처 목록 (sales + calendar)
FEATURE_COLS = ['sales', 'dow', 'dom', 'month', 'is_weekend', 'dow_sin', 'dow_cos']
TARGET_COL = 'sales'

print("Feature columns:", FEATURE_COLS)
df_feat.head(3)


Feature columns: ['sales', 'dow', 'dom', 'month', 'is_weekend', 'dow_sin', 'dow_cos']


,date_ordinal,date,store_menu,store,menu,sales,dow,dom,month,is_weekend,dow_sin,dow_cos
0,738521,2023-01-01,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0,6,1,1,1,-0.781831,0.62349
1,738522,2023-01-02,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0,0,2,1,0,0.000000,1.00000
2,738523,2023-01-03,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0,1,3,1,0,0.781831,0.62349


In [6]:
# ==== S1. Config & Seeds (folds 의존 제거) ====
import os, random, numpy as np, torch

class TrainCfg:
    def __init__(self):
        self.lookback = CFG.lookback
        self.horizon  = CFG.horizon
        self.patch_len = CFG.patch_len
        self.stride    = CFG.stride

        # training/tuning
        self.batch = 512
        self.wd = 2e-4
        self.grad_clip = 1.0
        self.use_amp = True
        self.swa_start_ratio = 0.6
        self.epochs_tune = 90
        self.epochs_final = 180

        self.seeds = [11,22,33,44,55]
        self.n_trials = 25
        self.tune_folds = 2   # <-- 기본값(임시). K-fold 만든 뒤 업데이트!

T = TrainCfg()

def set_seed(seed:int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


In [7]:
# ==== S2. store_menu ID & future calendar ====
SM_LIST = df_feat['store_menu'].drop_duplicates().tolist()
SM2ID = {sm: i+1 for i, sm in enumerate(SM_LIST)}  # 0: UNK
UNK_ID = 0

def future_cal_features(start_date, H=CFG.horizon):
    dates = pd.date_range(start_date, periods=H, freq='D')
    dow = dates.weekday
    dow_sin = np.sin(2*np.pi*dow/7)
    dow_cos = np.cos(2*np.pi*dow/7)
    is_weekend = (dow >= 5).astype(int)
    return np.concatenate([dow_sin, dow_cos, is_weekend], axis=0).astype(np.float32)  # len=H*3


In [8]:
# === 3. Sliding windows (28 -> 7) indexing ===
def build_samples_index(
    frame: pd.DataFrame,
    lookback: int,
    horizon: int,
    feature_cols: List[str],
    target_col: str = 'sales'
) -> pd.DataFrame:
    
    rows = []
    sample_id = 0

    for sm, g in frame.groupby('store_menu', sort=False):
        g = g.sort_values('date').reset_index(drop=True)
        n = len(g)
        # last valid target start index (inclusive)
        last_t = n - horizon
        # first valid target start index (input must fully exist)
        first_t = lookback
        for t in range(first_t, last_t + 1):
            inp_start = t - lookback
            inp_end   = t - 1
            tgt_start = t
            tgt_end   = t + horizon - 1

            rows.append({
                'sample_id': sample_id,
                'store_menu': sm,
                'input_start_idx': inp_start,
                'target_start_idx': tgt_start,
                'input_start_date': g.loc[inp_start, 'date'],
                'input_end_date':   g.loc[inp_end,   'date'],
                'target_start_date':g.loc[tgt_start, 'date'],
                'target_end_date':  g.loc[tgt_end,   'date'],
            })
            sample_id += 1

    samples = pd.DataFrame(rows).sort_values(['store_menu','target_start_date']).reset_index(drop=True)
    return samples

# 고유 ID 매핑
SM_LIST = df_feat['store_menu'].drop_duplicates().tolist()
SM2ID = {sm:i+1 for i, sm in enumerate(SM_LIST)}  # 0은 UNK
UNK_ID = 0

samples = build_samples_index(df_feat, CFG.lookback, CFG.horizon, FEATURE_COLS, TARGET_COL)
print("Total samples:", len(samples))
samples.head(3)

def future_cal_features(start_date, H=7):
    dates = pd.date_range(start_date, periods=H, freq='D')
    dow = dates.weekday
    dow_sin = np.sin(2*np.pi*dow/7)
    dow_cos = np.cos(2*np.pi*dow/7)
    is_weekend = (dow>=5).astype(int)
    # H×3 -> 평평하게(간단)
    return np.concatenate([dow_sin, dow_cos, is_weekend], axis=0).astype(np.float32)  # len=H*3



# Optional: PyTorch dataset skeleton for later training
if TORCH_AVAILABLE:
    class SalesWindowDataset(Dataset):
        def __init__(self, base_df, samples_df, feature_cols, target_col, lookback, horizon):
            self.base = base_df
            self.samples = samples_df.reset_index(drop=True)
            self.feat_cols = feature_cols
            self.tgt_col = target_col
            self.L, self.H = lookback, horizon
            self.group = {sm: g.reset_index(drop=True) for sm, g in self.base.groupby('store_menu', sort=False)}

        def __len__(self): return len(self.samples)

        def __getitem__(self, idx):
            s = self.samples.iloc[idx]
            g = self.group[s['store_menu']]
            inp_start = int(s['input_start_idx']); inp_end = inp_start + self.L
            tgt_start = int(s['target_start_idx']); tgt_end = tgt_start + self.H

            X = g.loc[inp_start:inp_end-1, self.feat_cols].to_numpy(np.float32)   # (L,C)
            y = g.loc[tgt_start:tgt_end-1, self.tgt_col].to_numpy(np.float32)     # (H,)
            sm_id = SM2ID.get(s['store_menu'], UNK_ID)
            fut_cal = future_cal_features(s['target_start_date'], H=self.H)        # (H*3,)

            return (torch.from_numpy(X), torch.from_numpy(y),
                    torch.tensor(sm_id, dtype=torch.long),
                    torch.from_numpy(fut_cal))



Total samples: 96114


In [9]:
# === 4. Time-aware K-fold with embargo (purged) ===
# === REPLACE: Time-aware K-fold with warm-up & safety ===
def build_time_kfold_splits_safe(
    samples: pd.DataFrame,
    n_splits: int,
    lookback: int,
    horizon: int,
    embargo_days: int,
    verbose: bool = True,
):
    s = samples.sort_values('target_start_date').reset_index(drop=True)

    # 1) 고유 검증 후보 날짜(실제 샘플이 존재하는 날짜만)
    uniq_dates = pd.Series(s['target_start_date'].unique()).sort_values().to_list()

    # 2) warm-up 이후만 검증으로 사용
    warmup = lookback + horizon + embargo_days   # 최소 70일 권장
    earliest_val_date = pd.Timestamp(uniq_dates[0]) + pd.Timedelta(days=warmup)
    val_date_candidates = [d for d in uniq_dates if d >= earliest_val_date]

    if len(val_date_candidates) < n_splits:
        if verbose:
            print(f"[Warn] 검증 후보 날짜가 {len(val_date_candidates)}일 뿐입니다. "
                  f"요청한 n_splits={n_splits} → {len(val_date_candidates)}로 축소.")
        n_splits = max(1, len(val_date_candidates))

    bins = np.array_split(np.array(val_date_candidates), n_splits)

    folds = []
    for k, val_dates in enumerate(bins, start=1):
        if len(val_dates) == 0:
            if verbose: print(f"[Skip] Fold {k}: 빈 검증 구간")
            continue

        val_start = pd.Timestamp(val_dates[0])
        val_end   = pd.Timestamp(val_dates[-1])
        val_mask  = s['target_start_date'].isin(val_dates)
        val_idx   = s.index[val_mask].to_numpy()

        # 학습은 검증 시작일 - embargo 이전의 것만
        cutoff = val_start - pd.Timedelta(days=embargo_days)
        train_mask = s['target_end_date'] < cutoff
        train_idx  = s.index[train_mask].to_numpy()

        if len(train_idx) == 0 or len(val_idx) == 0:
            if verbose:
                print(f"[Skip] Fold {k}: train={len(train_idx)}, val={len(val_idx)} (warm-up/embargo 과도) → 건너뜀")
            continue

        folds.append((train_idx, val_idx))
        if verbose:
            print(f"[Fold {len(folds)}/{n_splits}] "
                  f"Val {val_start.date()}→{val_end.date()} | "
                  f"train_size={len(train_idx):,}, val_size={len(val_idx):,}")

    # 안전장치: 남은 fold가 1개 이하라면 embargo를 완화해서 다시 시도
    if len(folds) <= 1:
        if verbose:
            print("[Info] 유효 fold가 너무 적습니다. embargo를 완화(예: lookback)하여 재시도합니다.")
        relaxed_embargo = max(lookback, embargo_days // 2)  # 최소 lookback
        return build_time_kfold_splits_safe(
            samples, n_splits, lookback, horizon, relaxed_embargo, verbose
        )

    return folds


folds = build_time_kfold_splits_safe(
    samples=samples,
    n_splits=CFG.n_splits,
    lookback=CFG.lookback,
    horizon=CFG.horizon,
    embargo_days=CFG.embargo_days,   # 35
    verbose=True
)


# (Optional) torch Dataset 예시 바인딩
if TORCH_AVAILABLE:
    full_dataset = SalesWindowDataset(
        base_df=df_feat,
        samples_df=samples,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        lookback=CFG.lookback,
        horizon=CFG.horizon
    )

    # Example: first fold indices
    tr_idx, va_idx = folds[0]
    print(f"First fold -> train:{len(tr_idx)}, val:{len(va_idx)}")


[Fold 1/5] Val 2023-04-09→2023-07-03 | train_size=5,597, val_size=16,598
[Fold 2/5] Val 2023-07-04→2023-09-27 | train_size=22,195, val_size=16,598
[Fold 3/5] Val 2023-09-28→2023-12-22 | train_size=38,793, val_size=16,598
[Fold 4/5] Val 2023-12-23→2024-03-16 | train_size=55,391, val_size=16,405
[Fold 5/5] Val 2024-03-17→2024-06-09 | train_size=71,796, val_size=16,405
First fold -> train:5597, val:16598


In [10]:
# ==== S1.5 update tune_folds (K-fold 생성 직후 실행) ====
T.tune_folds = min(T.tune_folds, len(folds))
print(f"[Tuning] use {T.tune_folds} fold(s) out of {len(folds)}")

[Tuning] use 2 fold(s) out of 5


In [11]:
# ==== S4. Loss & Naive ====
import torch.nn as nn
def smape_ignore_zero_torch(y_hat, y, eps=1e-6):
    mask = (y != 0).float()
    num = 2.0*torch.abs(y_hat - y)
    den = torch.abs(y_hat) + torch.abs(y) + eps
    return (num/den * mask).sum() / (mask.sum() + 1e-6)

def last_week_same_weekday_naive(x28: torch.Tensor):  # x28:(B,28)
    idx = torch.arange(28, device=x28.device)
    dow = (idx % 7)
    means = torch.stack([x28[:, dow==d].mean(dim=1) for d in range(7)], dim=1)  # (B,7)
    return means

class MixedLoss(nn.Module):
    def __init__(self, alpha=0.6, lam=0.15, horizon_weights=None):
        super().__init__()
        self.alpha, self.lam = alpha, lam
        self.h_w = None if horizon_weights is None else torch.tensor(horizon_weights).float().view(1,-1)
    def forward(self, y_hat, y, x28=None):
        eps=1e-6
        y_hat = torch.clamp(y_hat, min=0.0) + eps
        y     = torch.clamp(y,     min=0.0) + eps
        mae_log = torch.abs(torch.log1p(y_hat) - torch.log1p(y))
        sm = 2.0*torch.abs(y_hat - y)/(torch.abs(y_hat)+torch.abs(y)+eps)
        if self.h_w is not None:
            w = self.h_w.to(y_hat.device)
            mae_log = mae_log * w; sm = sm * w
        loss = 0.6*mae_log.mean() + 0.4*sm.mean()
        if (x28 is not None) and (self.lam>0):
            naive = last_week_same_weekday_naive(x28)
            loss += self.lam * torch.abs(torch.clamp(y_hat-eps, min=0.0) - naive).mean()
        return loss


In [12]:
# === 6. PatchTST (minimal) + RevIN — PATCHED ===
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==== S5. PatchTSTWeekly (last-token + weekly head + ID emb + cal bias) ====
import math, torch, torch.nn as nn

class RevIN(nn.Module):
    def __init__(self, eps=1e-5, min_std=1.0):
        super().__init__(); self.eps=eps; self.min_std=min_std
    def forward(self, x, sales_ch=0, stats=None, mode='norm'):
        if mode=='norm':
            mu = x.mean(dim=1, keepdim=True)
            sg = torch.clamp(x.std(dim=1, keepdim=True)+self.eps, min=self.min_std)
            return (x-mu)/sg, (mu[:,:,sales_ch], sg[:,:,sales_ch])
        elif mode=='denorm':
            mu_s, sg_s = stats; return x*sg_s + mu_s
        else: raise ValueError

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float()*(-math.log(10000.0)/d_model))
        pe[:,0::2] = torch.cos(pos*div); pe[:,1::2] = torch.sin(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1), :]

class PatchTSTWeekly(nn.Module):
    def __init__(self, *, lookback, horizon, c_in,
                 d_model=256, n_heads=8, depth=4,
                 patch_len=7, stride=1, dropout=0.2,
                 n_store_menu=1, emb_dim=32, fut_feat_dim=21, sales_ch=0):
        super().__init__()
        self.L, self.H, self.C = lookback, horizon, c_in
        self.patch_len, self.stride, self.sales_ch = patch_len, stride, sales_ch
        self.P = 1 + (self.L - self.patch_len)//self.stride

        self.embed = nn.Linear(self.patch_len*self.C, d_model)
        enc = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=depth)
        self.posenc = PositionalEncoding(d_model, max_len=self.P)

        self.sm_emb = nn.Embedding(n_store_menu+1, emb_dim)  # 0: UNK
        self.trunk_norm = nn.LayerNorm(d_model + emb_dim)

        self.w_head = nn.Sequential(nn.Linear(d_model+emb_dim, d_model), nn.ReLU(),
                                    nn.Linear(d_model,1), nn.Softplus())
        self.p_head = nn.Sequential(nn.Linear(d_model+emb_dim, d_model), nn.ReLU(),
                                    nn.Linear(d_model, self.H))
        self.cal_proj = nn.Linear(fut_feat_dim, self.H)

        self.revin = RevIN(min_std=1.0)

    def patchify(self, x):
        B,L,C = x.shape; chunks=[]
        for st in range(0, L-self.patch_len+1, self.stride):
            chunks.append(x[:, st:st+self.patch_len, :].reshape(B, -1))
        return torch.stack(chunks, dim=1)

    def forward(self, x, sm_id, fut_cal):
        x_n, stats = self.revin(x, sales_ch=self.sales_ch, mode='norm')
        z = self.embed(self.patchify(x_n))
        z = self.encoder(self.posenc(z))[:, -1, :]     # last token
        e = self.sm_emb(sm_id)
        z = self.trunk_norm(torch.cat([z, e], dim=-1))
        W = self.w_head(z).squeeze(-1)                 # (B,)
        logits = self.p_head(z) + self.cal_proj(fut_cal)  # (B,7)
        p = torch.softmax(logits, dim=-1)              # (B,7)
        y_hat_n = W.unsqueeze(1) * p                   # normalized
        return y_hat_n, stats




Device: cuda


In [13]:
# ==== S6. Train/Eval with SWA & AMP ====
from torch.utils.data import DataLoader, Subset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from copy import deepcopy

def build_dataloaders(train_idx, val_idx):
    ds_full = SalesWindowDataset(df_feat, samples, FEATURE_COLS, TARGET_COL, CFG.lookback, CFG.horizon)
    ds_tr, ds_va = Subset(ds_full, train_idx), Subset(ds_full, val_idx)
    dl_tr = DataLoader(ds_tr, batch_size=T.batch, shuffle=True,  num_workers=2, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=T.batch, shuffle=False, num_workers=2, pin_memory=True)
    return ds_tr, ds_va, dl_tr, dl_va

def train_one_fold_weekly(fold_id:int, seed:int, hps:dict):
    set_seed(seed)
    tr_idx, va_idx = folds[fold_id]
    ds_tr, ds_va, dl_tr, dl_va = build_dataloaders(tr_idx, va_idx)

    model = PatchTSTWeekly(
        lookback=CFG.lookback, horizon=CFG.horizon, c_in=len(FEATURE_COLS),
        d_model=hps['d_model'], n_heads=hps['n_heads'], depth=hps['depth'],
        patch_len=CFG.patch_len, stride=CFG.stride, dropout=hps['dropout'],
        n_store_menu=len(SM2ID), emb_dim=hps['emb_dim'], fut_feat_dim=CFG.horizon*3, sales_ch=0
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=hps['lr'], weight_decay=T.wd)
    base_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=40, T_mult=2)
    scaler = torch.cuda.amp.GradScaler(enabled=T.use_amp)

    swa_start = int(T.swa_start_ratio * hps['epochs'])
    swa_model = AveragedModel(model)
    swa_scheduler = SWALR(opt, swa_lr=hps['lr']*0.5)

    loss_fn = MixedLoss(alpha=hps['alpha'], lam=hps['lambda'], horizon_weights=hps.get('h_w', None))
    best = 1e9; best_sd = None

    for ep in range(1, hps['epochs']+1):
        model.train()
        for X,y,sm_id,fcal in dl_tr:
            X,y,sm_id,fcal = X.to(device), y.to(device), sm_id.to(device), fcal.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=T.use_amp):
                y_hat_n, stats = model(X, sm_id, fcal)
                y_hat = model.revin(y_hat_n, stats=stats, mode='denorm').clamp(min=0.0)
                x28 = X[:,:,0]
                loss = loss_fn(y_hat, y, x28=x28)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), T.grad_clip)
            scaler.step(opt); scaler.update()
        if ep >= swa_start:
            swa_model.update_parameters(model); swa_scheduler.step()
        else:
            base_scheduler.step()

        # ---- validation
        model.eval()
        with torch.no_grad():
            Y=[]; P=[]
            for Xv,yv,sm_idv,fcalv in dl_va:
                Xv,yv,sm_idv,fcalv = Xv.to(device), yv.to(device), sm_idv.to(device), fcalv.to(device)
                y_hat_n, stats = model(Xv, sm_idv, fcalv)
                y_hat = model.revin(y_hat_n, stats=stats, mode='denorm').clamp(min=0.0)
                Y.append(yv); P.append(y_hat)
            Y = torch.cat(Y); P = torch.cat(P)
            val_smape = smape_ignore_zero_torch(P, Y).item()

        if val_smape < best-1e-6:
            best = val_smape; best_sd = deepcopy(model.state_dict())
        if ep % 10 == 0:
            print(f"[Fold{fold_id} Seed{seed}] Ep{ep}/{hps['epochs']} | val_sMAPE={val_smape:.4f} | best={best:.4f}")

    # SWA finalize
    update_bn(dl_tr, swa_model, device=device)
    swa_model.eval()
    with torch.no_grad():
        Y=[]; P=[]
        for Xv,yv,sm_idv,fcalv in dl_va:
            Xv,yv,sm_idv,fcalv = Xv.to(device), yv.to(device), sm_idv.to(device), fcalv.to(device)
            y_hat_n, stats = swa_model(Xv, sm_idv, fcalv)
            y_hat = model.revin(y_hat_n, stats=stats, mode='denorm').clamp(min=0.0)
            Y.append(yv); P.append(y_hat)
        swa_smape = smape_ignore_zero_torch(torch.cat(P), torch.cat(Y)).item()

    use_swa = swa_smape < best
    final_sd = swa_model.state_dict() if use_swa else best_sd
    ckpt = f"./weekly_fold{fold_id}_seed{seed}.pt"
    torch.save(final_sd, ckpt)
    print(f"[Fold{fold_id} Seed{seed}] Saved -> {ckpt} | best={min(best, swa_smape):.4f}")
    return min(best, swa_smape), ckpt


In [14]:
# ==== S7. Optuna objective (safe) ====
import optuna, numpy as np
from optuna.exceptions import TrialPruned

def objective(trial: optuna.trial.Trial):
    # 1) d_model 먼저 선택
    d_model = trial.suggest_categorical('d_model', [192, 256, 320])

    # 2) d_model을 나누는 head만 허용 (깊게 갈 필요 없으면 [4,8,16] 고정 권장)
    head_candidates = [4, 8, 16]  # <- 간단/안전
    valid_heads = [h for h in head_candidates if d_model % h == 0]
    if not valid_heads:
        # (사실 위 후보에선 비지 않지만 방어)
        valid_heads = [8]

    hps = {
        'd_model': d_model,
        'n_heads': trial.suggest_categorical('n_heads', valid_heads),
        'depth'  : trial.suggest_categorical('depth', [3, 4, 5]),
        'dropout': trial.suggest_float('dropout', 0.1, 0.3),
        'emb_dim': trial.suggest_categorical('emb_dim', [16, 32, 48]),
        'lr'     : trial.suggest_float('lr', 3e-4, 3e-3, log=True),
        'alpha'  : trial.suggest_float('alpha', 0.5, 0.75),
        'lambda' : trial.suggest_float('lambda', 0.0, 0.25),
        'epochs' : T.epochs_tune,
        'h_w'    : [1,1,1,1,1.2,1.3,1.5] if trial.suggest_categorical('use_h_weights',[0,1])==1 else None
    }

    try:
        seed = 2025
        scores = []
        for k in range(min(T.tune_folds, len(folds))):
            ckpt_path = f"./weekly_fold{k}_seed2025.pt"
            if os.path.exists(ckpt_path):
                print(f"[Optuna] Skip Fold{k} (checkpoint exists)")
                # ckpt에서 점수 읽어오기 (optional)
                best_score = torch.load(ckpt_path, map_location='cpu').get('best', None)
                if best_score is not None:
                    scores.append(float(best_score))
                    continue
            # 체크포인트 없으면 학습 실행
            s, _ = train_one_fold_weekly(k, seed, hps)
            scores.append(s)
        return float(np.mean(scores))
    except AssertionError as e:
        # 잘못된 조합 등은 해당 trial만 가지치기
        raise TrialPruned(str(e))
    except Exception as e:
        print("[Optuna] Pruned trial due to exception:", repr(e))
        raise TrialPruned(repr(e))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=T.n_trials, show_progress_bar=True)

best_hps = study.best_trial.params
best_hps['epochs'] = T.epochs_final
best_hps['h_w'] = [1,1,1,1,1.2,1.3,1.5] if best_hps.get('use_h_weights',0)==1 else None
print("Best sMAPE:", study.best_value, "\nBest hps:", best_hps)


[I 2025-08-08 21:19:24,145] A new study created in memory with name: no-name-5abbdf2e-3789-4830-939c-ec1643c10add
  0%|          | 0/25 [00:00<?, ?it/s]

[Optuna] Skip Fold0 (checkpoint exists)
[Fold0 Seed2025] Ep10/90 | val_sMAPE=0.6782 | best=0.6779
[Fold0 Seed2025] Ep20/90 | val_sMAPE=0.6782 | best=0.6779
[Fold0 Seed2025] Ep30/90 | val_sMAPE=0.6782 | best=0.6779
[Fold0 Seed2025] Ep40/90 | val_sMAPE=0.6782 | best=0.6779
[Fold0 Seed2025] Ep50/90 | val_sMAPE=0.6782 | best=0.6779
[Fold0 Seed2025] Ep60/90 | val_sMAPE=0.6782 | best=0.6779
[Fold0 Seed2025] Ep70/90 | val_sMAPE=0.6782 | best=0.6779
[Fold0 Seed2025] Ep80/90 | val_sMAPE=0.6782 | best=0.6779
[Fold0 Seed2025] Ep90/90 | val_sMAPE=0.6782 | best=0.6779
[Fold0 Seed2025] Saved -> ./weekly_fold0_seed2025.pt | best=0.6779
[Optuna] Skip Fold1 (checkpoint exists)
[Fold1 Seed2025] Ep10/90 | val_sMAPE=0.6811 | best=0.6811
[Fold1 Seed2025] Ep20/90 | val_sMAPE=0.6811 | best=0.6811
[Fold1 Seed2025] Ep30/90 | val_sMAPE=0.6811 | best=0.6811
[Fold1 Seed2025] Ep40/90 | val_sMAPE=0.6811 | best=0.6811
[Fold1 Seed2025] Ep50/90 | val_sMAPE=0.6811 | best=0.6811
[Fold1 Seed2025] Ep60/90 | val_sMAPE=0.68

Best trial: 0. Best value: 0.679479:   4%|▍         | 1/25 [55:46<22:18:31, 3346.30s/it]

[Fold1 Seed2025] Saved -> ./weekly_fold1_seed2025.pt | best=0.6811
[I 2025-08-08 22:15:10,445] Trial 0 finished with value: 0.6794787049293518 and parameters: {'d_model': 192, 'n_heads': 16, 'depth': 3, 'dropout': 0.16103149116339194, 'emb_dim': 32, 'lr': 0.0005500531768790179, 'alpha': 0.7392952814160284, 'lambda': 0.04630826275829486, 'use_h_weights': 0}. Best is trial 0 with value: 0.6794787049293518.
[Optuna] Skip Fold0 (checkpoint exists)
[Fold0 Seed2025] Ep10/90 | val_sMAPE=0.6782 | best=0.6782
[Fold0 Seed2025] Ep20/90 | val_sMAPE=0.6782 | best=0.6782
[Fold0 Seed2025] Ep30/90 | val_sMAPE=0.6782 | best=0.6782
[Fold0 Seed2025] Ep40/90 | val_sMAPE=0.6782 | best=0.6782
[Fold0 Seed2025] Ep50/90 | val_sMAPE=0.6782 | best=0.6782
[Fold0 Seed2025] Ep60/90 | val_sMAPE=0.6782 | best=0.6782
[Fold0 Seed2025] Ep70/90 | val_sMAPE=0.6782 | best=0.6782
[Fold0 Seed2025] Ep80/90 | val_sMAPE=0.6782 | best=0.6782
[Fold0 Seed2025] Ep90/90 | val_sMAPE=0.6782 | best=0.6782
[Fold0 Seed2025] Saved -> ./we

Best trial: 0. Best value: 0.679479:   4%|▍         | 1/25 [1:20:35<32:14:10, 4835.43s/it]
Exception in thread Thread-567 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 61, in _pin_memory_loop
    do_one_step()
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 37, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python

[W 2025-08-08 22:39:59,570] Trial 1 failed with parameters: {'d_model': 320, 'n_heads': 16, 'depth': 3, 'dropout': 0.2615412037714009, 'emb_dim': 48, 'lr': 0.0009452879546577451, 'alpha': 0.516305927017632, 'lambda': 0.07133346427301776, 'use_h_weights': 0} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_757287/3656762583.py", line 42, in objective
    s, _ = train_one_fold_weekly(k, seed, hps)
  File "/tmp/ipykernel_757287/1678518716.py", line 58, in train_one_fold_weekly
    for Xv,yv,sm_idv,fcalv in dl_va:
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 734, in __next__
    data = self._next_data()
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/site-packages/torch/utils/data/dataloader.py

    fd = df.detach()
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/multiprocessing/resource_sharer.py", line 58, in detach
    return reduction.recv_handle(conn)
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/multiprocessing/reduction.py", line 189, in recv_handle
    return recvfds(s, 1)[0]
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/multiprocessing/reduction.py", line 157, in recvfds
    msg, ancdata, flags, addr = sock.recvmsg(1, socket.CMSG_SPACE(bytes_size))
ConnectionResetError: [Errno 104] Connection reset by peer


KeyboardInterrupt: 

In [ ]:
# ==== S8. Final training with seeds & all folds ====
fold_ckpts, fold_scores = [], []
for seed in T.seeds:
    for k in range(len(folds)):
        s, ck = train_one_fold_weekly(k, seed, best_hps)
        fold_scores.append(s); fold_ckpts.append(ck)
print("Avg sMAPE:", np.mean(fold_scores))
len(fold_ckpts), fold_ckpts[:3]


In [ ]:
# ==== S9. Inference with naive blend (rho=0.2) ====
import glob
RHO = 0.2

@torch.no_grad()
def predict_7days_weekly(test_path: str, ckpt_paths: list, save_path: str):
    test = pd.read_csv(test_path)
    test['date'] = pd.to_datetime(test['date'])
    if 'store_menu' not in test.columns:
        test['store_menu'] = test['store'].astype(str) + "_" + test['menu'].astype(str)
    test = test.sort_values(['store_menu','date']).reset_index(drop=True)
    feat = add_calendar_features(test)

    last_date = feat['date'].max()
    target_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=CFG.horizon, freq='D')

    # 모델 로드
    models=[]
    for ck in ckpt_paths:
        m = PatchTSTWeekly(
            lookback=CFG.lookback, horizon=CFG.horizon, c_in=len(FEATURE_COLS),
            d_model=best_hps['d_model'], n_heads=best_hps['n_heads'], depth=best_hps['depth'],
            patch_len=CFG.patch_len, stride=CFG.stride, dropout=best_hps['dropout'],
            n_store_menu=len(SM2ID), emb_dim=best_hps['emb_dim'], fut_feat_dim=CFG.horizon*3, sales_ch=0
        ).to(device)
        sd = torch.load(ck, map_location=device)
        m.load_state_dict(sd, strict=True); m.eval()
        models.append(m)

    preds={}
    for sm, g in feat.groupby('store_menu', sort=False):
        g_last = g.tail(CFG.lookback)
        X = torch.from_numpy(g_last[FEATURE_COLS].to_numpy(np.float32)).unsqueeze(0).to(device)  # (1,28,C)
        sm_id = torch.tensor([SM2ID.get(sm, 0)], dtype=torch.long, device=device)
        fcal = torch.from_numpy(future_cal_features(last_date + pd.Timedelta(days=1))).unsqueeze(0).to(device)

        outs=[]
        for m in models:
            y_hat_n, stats = m(X, sm_id, fcal)
            y_hat = m.revin(y_hat_n, stats=stats, mode='denorm').clamp(min=0.0)
            outs.append(y_hat)
        pred = torch.mean(torch.stack(outs, dim=0), dim=0).squeeze(0)  # (7,)

        naive = last_week_same_weekday_naive(X[:,:,0]).squeeze(0)
        pred_final = (1 - RHO) * pred + RHO * naive
        preds[sm] = pred_final.cpu().numpy()

    sm_list = list(feat['store_menu'].drop_duplicates())
    sub = pd.DataFrame(index=target_dates, columns=sm_list, dtype=float)
    for sm in sm_list: sub[sm] = preds[sm]
    sub.index.name = 'date'
    sub.to_csv(save_path); print(f"[Saved] {save_path} | shape={sub.shape}")
    return sub

# 실행
test_files = sorted(glob.glob("./data_filtering/filtered/TEST_0*.csv"))
print("Found:", test_files)
for tp in test_files:
    tag = os.path.splitext(os.path.basename(tp))[0]
    _ = predict_7days_weekly(tp, fold_ckpts, f"./submission_{tag}.csv")


Found test files: ['./data_filtering/filtered/TEST_00.csv', './data_filtering/filtered/TEST_01.csv', './data_filtering/filtered/TEST_02.csv', './data_filtering/filtered/TEST_03.csv', './data_filtering/filtered/TEST_04.csv', './data_filtering/filtered/TEST_05.csv', './data_filtering/filtered/TEST_06.csv', './data_filtering/filtered/TEST_07.csv', './data_filtering/filtered/TEST_08.csv', './data_filtering/filtered/TEST_09.csv']


/tmp/ipykernel_419992/1639769317.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_00.csv | shape=(7, 193)


/tmp/ipykernel_419992/1639769317.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_01.csv | shape=(7, 193)


...

[Saved] ./submission_TEST_07.csv | shape=(7, 193)


/tmp/ipykernel_419992/1639769317.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_08.csv | shape=(7, 193)


/tmp/ipykernel_419992/1639769317.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_09.csv | shape=(7, 193)


In [ ]:
import pandas as pd
import glob

# submission_TEST_00.csv ~ submission_TEST_09.csv 파일을 번호 순서대로 불러와서 하나의 데이터프레임으로 합치기
submission_files = sorted(glob.glob("./submission_TEST_0*.csv"), key=lambda x: int(x.split("_")[-1].split(".")[0]))
dfs = []
for file in submission_files:
    df = pd.read_csv(file, index_col=0)
    dfs.append(df)
merged_submission = pd.concat(dfs, axis=0)

merged_submission.to_csv("merged_submission.csv")
print("merged_submission.csv 파일로 저장 완료")


merged_submission.csv 파일로 저장 완료
